# 📊 Clinisys Embrioes Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database (`huntington_data_lake.duckdb`) and **AWS Athena (Production)** (`gold_huntington_prod`) to execute data quality and validation queries on the `clinisys_embrioes` table.

### Key Metrics Reconciled:
1. **Total Row Count (`total_rows`)**
2. **Unique Oocito IDs (`unique_oocito_id`) & Duplicate Check**
3. **Presence of Linked Entity IDs**:
   - `trat1_id` (Primary Treatment)
   - `emb_cong_id` (Frozen Embryo)
   - `cong_em_id` (Freezing Procedure)
   - `descong_em_id` (Thawing Procedure)
   - `trat2_id` (Transfer Treatment)

### Breakdowns & Analyses Included:
- **Part 1 & 2**: DuckDB & Athena Overall, Per Year (`micro_data_procedimento`), and Per Unidade (`patient_info.unidade_nome`)
- **Part 3**: Side-by-Side Reconciliation Summary Table
- **Part 4**: Discrepancy Analysis — Oocito IDs present ONLY in DuckDB or ONLY in Athena
- **Part 5**: Attribute Discrepancy Analysis — Per-key comparison (`trat1_id`, `emb_cong_id`, `cong_em_id`, `descong_em_id`, `trat2_id`, `oocito_embryo_number`) for matching oocitos
- **Part 6**: PGT / Genetic Test Results Side-by-Side Comparison (`oocito_resultado_pgd`, normalized NULLs/empty)
- **Part 7**: Date Sequence & Timeline Integrity Side-by-Side (`micro_data_procedimento`, `cong_em_data`, `descong_em_data_transferencia`)
- **Part 8**: Treatment Outcomes & Non-Transfer Reasons Side-by-Side (`trat1_resultado_tratamento`, `trat1_motivo_nao_transferir`)


In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_SCHEMA = 'gold_huntington_prod'
STAGING_SCHEMA = 'gold_huntington_staging'

print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print(f"Connecting to AWS Athena ({ATHENA_SCHEMA})...")
try:
    ath_con = connect(
        region_name=ATHENA_REGION,
        work_group=ATHENA_WORKGROUP,
        schema_name=ATHENA_SCHEMA
    )
    ath_cur = ath_con.cursor()
    print("Successfully connected to AWS Athena.")
except Exception as e:
    print(f"Warning: Could not connect to Athena: {e}")
    ath_cur = None


## 🔍 Part 1: DuckDB Queries (Local Gold Layer)

Executing local validation queries on `gold.clinisys_embrioes` and `gold.patient_info` in DuckDB.


In [ ]:
# 1. Overall Totals (DuckDB)
query_duck_overall = '''
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
    ROUND((COUNT(*) - COUNT(DISTINCT oocito_id)) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_duplicate_oocito_id,
    COUNT(trat1_id) as count_trat1_id,
    ROUND(COUNT(trat1_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
    COUNT(emb_cong_id) as count_emb_cong_id,
    ROUND(COUNT(emb_cong_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
    COUNT(cong_em_id) as count_cong_em_id,
    ROUND(COUNT(cong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
    COUNT(descong_em_id) as count_descong_em_id,
    ROUND(COUNT(descong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
    COUNT(trat2_id) as count_trat2_id,
    ROUND(COUNT(trat2_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
FROM gold.clinisys_embrioes
'''
df_duck_overall = duck_con.execute(query_duck_overall).df()
print("--- DuckDB Overall Totals ---")
display(df_duck_overall)

# 2. Per Year Breakdown (DuckDB)
query_duck_year = '''
SELECT 
    YEAR(CAST(micro_data_procedimento AS DATE)) as procedimento_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
    COUNT(trat1_id) as count_trat1_id,
    ROUND(COUNT(trat1_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
    COUNT(emb_cong_id) as count_emb_cong_id,
    ROUND(COUNT(emb_cong_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
    COUNT(cong_em_id) as count_cong_em_id,
    ROUND(COUNT(cong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
    COUNT(descong_em_id) as count_descong_em_id,
    ROUND(COUNT(descong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
    COUNT(trat2_id) as count_trat2_id,
    ROUND(COUNT(trat2_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
FROM gold.clinisys_embrioes
WHERE micro_data_procedimento IS NOT NULL
GROUP BY YEAR(CAST(micro_data_procedimento AS DATE))
ORDER BY procedimento_year DESC
'''
df_duck_year = duck_con.execute(query_duck_year).df()
print("\n--- DuckDB Per Year Breakdown ---")
display(df_duck_year)

# 3. Per Unidade Breakdown (DuckDB)
query_duck_unidade = '''
SELECT 
    COALESCE(p.unidade_nome, 'NA / Desconhecido') as unidade_nome,
    COUNT(*) as total_rows,
    COUNT(DISTINCT c.oocito_id) as unique_oocito_id,
    COUNT(*) - COUNT(DISTINCT c.oocito_id) as duplicate_oocito_id_count,
    COUNT(c.trat1_id) as count_trat1_id,
    ROUND(COUNT(c.trat1_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
    COUNT(c.emb_cong_id) as count_emb_cong_id,
    ROUND(COUNT(c.emb_cong_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
    COUNT(c.cong_em_id) as count_cong_em_id,
    ROUND(COUNT(c.cong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
    COUNT(c.descong_em_id) as count_descong_em_id,
    ROUND(COUNT(c.descong_em_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
    COUNT(c.trat2_id) as count_trat2_id,
    ROUND(COUNT(c.trat2_id) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
FROM gold.clinisys_embrioes c
LEFT JOIN gold.patient_info p ON c.micro_prontuario = p.prontuario
GROUP BY COALESCE(p.unidade_nome, 'NA / Desconhecido')
ORDER BY total_rows DESC
'''
df_duck_unidade = duck_con.execute(query_duck_unidade).df()
print("\n--- DuckDB Per Unidade Breakdown ---")
display(df_duck_unidade)


## ☁️ Part 2: AWS Athena Queries (Production Gold Layer)

Executing queries against AWS Athena (`{ATHENA_SCHEMA}.clinisys_embrioes`).


In [ ]:
# Athena Queries Execution
if ath_cur is not None:
    # 1. Overall Totals (Athena)
    query_ath_overall = f'''
    SELECT 
        'TOTAL' as grouping_type,
        'ALL' as grouping_value,
        COUNT(*) as total_rows,
        COUNT(DISTINCT oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
        ROUND(CAST(COUNT(*) - COUNT(DISTINCT oocito_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_duplicate_oocito_id,
        COUNT(trat1_id) as count_trat1_id,
        ROUND(CAST(COUNT(trat1_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
        COUNT(emb_cong_id) as count_emb_cong_id,
        ROUND(CAST(COUNT(emb_cong_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
        COUNT(cong_em_id) as count_cong_em_id,
        ROUND(CAST(COUNT(cong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
        COUNT(descong_em_id) as count_descong_em_id,
        ROUND(CAST(COUNT(descong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
        COUNT(trat2_id) as count_trat2_id,
        ROUND(CAST(COUNT(trat2_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    '''
    try:
        ath_cur.execute(query_ath_overall)
        df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("--- AWS Athena Overall Totals ---")
        display(df_ath_overall)
    except Exception as e:
        print(f"Error running Athena overall query: {e}")
        df_ath_overall = pd.DataFrame()

    # 2. Per Year Breakdown (Athena)
    query_ath_year = f'''
    SELECT 
        YEAR(CAST(micro_data_procedimento AS DATE)) as procedimento_year,
        COUNT(*) as total_rows,
        COUNT(DISTINCT oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT oocito_id) as duplicate_oocito_id_count,
        COUNT(trat1_id) as count_trat1_id,
        ROUND(CAST(COUNT(trat1_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
        COUNT(emb_cong_id) as count_emb_cong_id,
        ROUND(CAST(COUNT(emb_cong_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
        COUNT(cong_em_id) as count_cong_em_id,
        ROUND(CAST(COUNT(cong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
        COUNT(descong_em_id) as count_descong_em_id,
        ROUND(CAST(COUNT(descong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
        COUNT(trat2_id) as count_trat2_id,
        ROUND(CAST(COUNT(trat2_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    WHERE micro_data_procedimento IS NOT NULL
    GROUP BY YEAR(CAST(micro_data_procedimento AS DATE))
    ORDER BY procedimento_year DESC
    '''
    try:
        ath_cur.execute(query_ath_year)
        df_ath_year = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- AWS Athena Per Year Breakdown ---")
        display(df_ath_year)
    except Exception as e:
        print(f"Error running Athena year query: {e}")
        df_ath_year = pd.DataFrame()

    # 3. Per Unidade Breakdown (Athena - joining with gold_huntington_staging.patient_info)
    query_ath_unidade = f'''
    SELECT 
        COALESCE(p.unidade_nome, 'NA / Desconhecido') as unidade_nome,
        COUNT(*) as total_rows,
        COUNT(DISTINCT c.oocito_id) as unique_oocito_id,
        COUNT(*) - COUNT(DISTINCT c.oocito_id) as duplicate_oocito_id_count,
        COUNT(c.trat1_id) as count_trat1_id,
        ROUND(CAST(COUNT(c.trat1_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat1_id,
        COUNT(c.emb_cong_id) as count_emb_cong_id,
        ROUND(CAST(COUNT(c.emb_cong_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_emb_cong_id,
        COUNT(c.cong_em_id) as count_cong_em_id,
        ROUND(CAST(COUNT(c.cong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_cong_em_id,
        COUNT(c.descong_em_id) as count_descong_em_id,
        ROUND(CAST(COUNT(c.descong_em_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_descong_em_id,
        COUNT(c.trat2_id) as count_trat2_id,
        ROUND(CAST(COUNT(c.trat2_id) AS DOUBLE) * 100.0 / NULLIF(COUNT(*), 0), 2) as pct_trat2_id
    FROM {ATHENA_SCHEMA}.clinisys_embrioes c
    LEFT JOIN {STAGING_SCHEMA}.patient_info p ON c.micro_prontuario = p.prontuario
    GROUP BY COALESCE(p.unidade_nome, 'NA / Desconhecido')
    ORDER BY total_rows DESC
    '''
    try:
        ath_cur.execute(query_ath_unidade)
        df_ath_unidade = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- AWS Athena Per Unidade Breakdown ---")
        display(df_ath_unidade)
    except Exception as e:
        print(f"Error running Athena unidade query: {e}")
        df_ath_unidade = pd.DataFrame()
else:
    print("Athena cursor unavailable. Skipping Athena queries.")
    df_ath_overall = pd.DataFrame()
    df_ath_year = pd.DataFrame()
    df_ath_unidade = pd.DataFrame()


## ⚖️ Part 3: Side-by-Side Reconciliation & Delta Comparison

Compares DuckDB vs Athena totals and highlights differences.


In [ ]:
# Build Reconciliation Summary
if not df_duck_overall.empty:
    duck_summary = df_duck_overall.iloc[0].to_dict()
    
    if not df_ath_overall.empty:
        ath_summary = df_ath_overall.iloc[0].to_dict()
    else:
        ath_summary = {k: None for k in duck_summary.keys()}
        
    metrics = [
        'total_rows',
        'unique_oocito_id',
        'duplicate_oocito_id_count',
        'count_trat1_id',
        'count_emb_cong_id',
        'count_cong_em_id',
        'count_descong_em_id',
        'count_trat2_id'
    ]
    
    recon_rows = []
    for m in metrics:
        v_duck = duck_summary.get(m, 0)
        v_ath = ath_summary.get(m, 0) if ath_summary.get(m) is not None else 0
        diff = (v_duck - v_ath) if (v_duck is not None and v_ath is not None) else None
        pct_match = (100.0 * min(v_duck, v_ath) / max(v_duck, v_ath)) if (v_duck and v_ath) else (100.0 if v_duck == v_ath else 0.0)
        
        recon_rows.append({
            'Metric': m,
            'DuckDB (Local)': v_duck,
            'AWS Athena (Prod)': v_ath,
            'Delta (Duck - Ath)': diff,
            'Match %': f"{pct_match:.2f}%" if pct_match is not None else 'N/A'
        })
        
    df_recon = pd.DataFrame(recon_rows)
    print("=========================================================================")
    print("               SUMMARY RECONCILIATION: DUCKDB VS ATHENA                  ")
    print("=========================================================================")
    display(df_recon)


## 🔎 Part 4: Oocito Discrepancy Analysis (Missing Records)

Identifies `oocito_id`s that exist **exclusively in DuckDB** or **exclusively in AWS Athena**, displaying sample records for detailed investigation.


In [ ]:
# Part 4: Discrepancy Samples
print("Fetching set of unique oocito_ids from DuckDB...")
duck_oocito_ids = set(duck_con.execute("SELECT DISTINCT oocito_id FROM gold.clinisys_embrioes WHERE oocito_id IS NOT NULL").df()['oocito_id'])
print(f"Total unique oocito_ids in DuckDB: {len(duck_oocito_ids):,}")

if ath_cur is not None:
    try:
        print(f"Fetching set of unique oocito_ids from Athena ({ATHENA_SCHEMA})...")
        ath_cur.execute(f"SELECT DISTINCT oocito_id FROM {ATHENA_SCHEMA}.clinisys_embrioes WHERE oocito_id IS NOT NULL")
        ath_oocito_ids = set(r[0] for r in ath_cur.fetchall())
        print(f"Total unique oocito_ids in Athena: {len(ath_oocito_ids):,}")
    except Exception as e:
        print(f"Error querying Athena for oocito_ids: {e}")
        ath_oocito_ids = set()
else:
    print("Athena connection unavailable. Skipping Athena oocito set fetching.")
    ath_oocito_ids = set()

# Set difference logic
only_in_duck = list(duck_oocito_ids - ath_oocito_ids) if ath_oocito_ids else []
only_in_ath = list(ath_oocito_ids - duck_oocito_ids) if ath_oocito_ids else []

print("\n=========================================================================")
print(f"               DISCREPANCY SUMMARY: OOCITO_ID SET DIFFERENCE             ")
print("=========================================================================")
print(f"1. Oocito IDs present ONLY in DuckDB: {len(only_in_duck):,}")
print(f"2. Oocito IDs present ONLY in Athena: {len(only_in_ath):,}")

# Sample 1: Oocitos ONLY in DuckDB
if only_in_duck:
    sample_duck_ids = tuple(only_in_duck[:100])
    in_clause = f"({sample_duck_ids[0]})" if len(sample_duck_ids) == 1 else str(sample_duck_ids)
    
    query_sample_duck = f'''
    SELECT 
        c.oocito_id, 
        c.micro_prontuario, 
        c.micro_data_procedimento, 
        c.trat1_id, 
        c.emb_cong_id, 
        c.descong_em_id, 
        c.nome_medico,
        p.unidade_nome
    FROM gold.clinisys_embrioes c
    LEFT JOIN gold.patient_info p ON c.micro_prontuario = p.prontuario
    WHERE c.oocito_id IN {in_clause}
    LIMIT 15
    '''
    df_sample_duck = duck_con.execute(query_sample_duck).df()
    print("\n--- Sample Oocitos ONLY in DuckDB (Local) ---")
    display(df_sample_duck)
else:
    print("\nNo oocitos found exclusively in DuckDB (or Athena set not fetched).")

# Sample 2: Oocitos ONLY in Athena
if only_in_ath and ath_cur is not None:
    sample_ath_ids = tuple(only_in_ath[:100])
    in_clause_ath = f"({sample_ath_ids[0]})" if len(sample_ath_ids) == 1 else str(sample_ath_ids)
    
    query_sample_ath = f'''
    SELECT 
        c.oocito_id, 
        c.micro_prontuario, 
        c.micro_data_procedimento, 
        c.trat1_id, 
        c.emb_cong_id, 
        c.descong_em_id, 
        c.nome_medico,
        p.unidade_nome
    FROM {ATHENA_SCHEMA}.clinisys_embrioes c
    LEFT JOIN {STAGING_SCHEMA}.patient_info p ON c.micro_prontuario = p.prontuario
    WHERE c.oocito_id IN {in_clause_ath}
    LIMIT 15
    '''
    try:
        ath_cur.execute(query_sample_ath)
        df_sample_ath = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\n--- Sample Oocitos ONLY in AWS Athena (Prod) ---")
        display(df_sample_ath)
    except Exception as e:
        print(f"Error querying sample oocitos from Athena: {e}")
else:
    print("\nNo oocitos found exclusively in AWS Athena (or Athena set not fetched).")


## 🔀 Part 5: Attribute Discrepancy Analysis (Per Key Attribute)

Compares matching `oocito_id`s across DuckDB and Athena to find records where specific key attributes differ.
Key attributes evaluated:
1. `trat1_id`
2. `emb_cong_id`
3. `cong_em_id`
4. `descong_em_id`
5. `trat2_id`
6. `oocito_embryo_number`


In [ ]:
# Part 5 Data Preparation: Fetch matching oocito records for attribute comparison
print("Fetching key attribute columns from DuckDB...")
df_keys_duck = duck_con.execute('''
    SELECT 
        oocito_id, 
        micro_prontuario,
        micro_data_procedimento,
        nome_medico,
        trat1_id, 
        emb_cong_id, 
        cong_em_id, 
        descong_em_id, 
        trat2_id, 
        oocito_embryo_number
    FROM gold.clinisys_embrioes
    WHERE oocito_id IS NOT NULL
''').df()

if ath_cur is not None:
    try:
        print(f"Fetching key attribute columns from Athena ({ATHENA_SCHEMA})...")
        ath_cur.execute(f'''
            SELECT 
                oocito_id, 
                micro_prontuario,
                micro_data_procedimento,
                nome_medico,
                trat1_id, 
                emb_cong_id, 
                cong_em_id, 
                descong_em_id, 
                trat2_id, 
                oocito_embryo_number
            FROM {ATHENA_SCHEMA}.clinisys_embrioes
            WHERE oocito_id IS NOT NULL
        ''')
        df_keys_ath = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print(f"Loaded {len(df_keys_duck):,} DuckDB rows and {len(df_keys_ath):,} Athena rows.")
    except Exception as e:
        print(f"Error fetching Athena key columns: {e}")
        df_keys_ath = pd.DataFrame()
else:
    print("Athena cursor unavailable. Skipping Athena key comparison.")
    df_keys_ath = pd.DataFrame()

# Deduplicate on oocito_id for clean 1-to-1 attribute comparison
df_duck_unique = df_keys_duck.drop_duplicates(subset=['oocito_id'])
df_ath_unique = df_keys_ath.drop_duplicates(subset=['oocito_id']) if not df_keys_ath.empty else pd.DataFrame()

if not df_duck_unique.empty and not df_ath_unique.empty:
    df_merged_keys = pd.merge(
        df_duck_unique, 
        df_ath_unique, 
        on='oocito_id', 
        suffixes=('_duck', '_ath')
    )
    print(f"Total matching oocito_ids for attribute comparison: {len(df_merged_keys):,}")
else:
    df_merged_keys = pd.DataFrame()


### 🔑 Key Comparison: `trat1_id`

In [ ]:
# Discrepancy Check for Key: trat1_id
if not df_merged_keys.empty:
    col_duck = 'trat1_id_duck'
    col_ath = 'trat1_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck != v_ath) & ~(v_duck.isna() & v_ath.isna())
    df_diff_trat1_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_trat1_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: trat1_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_trat1_id[display_cols].rename(columns={
            col_duck: f'trat1_id_duckdb',
            col_ath: f'trat1_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'trat1_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


### 🔑 Key Comparison: `emb_cong_id`

In [ ]:
# Discrepancy Check for Key: emb_cong_id
if not df_merged_keys.empty:
    col_duck = 'emb_cong_id_duck'
    col_ath = 'emb_cong_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck != v_ath) & ~(v_duck.isna() & v_ath.isna())
    df_diff_emb_cong_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_emb_cong_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: emb_cong_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_emb_cong_id[display_cols].rename(columns={
            col_duck: f'emb_cong_id_duckdb',
            col_ath: f'emb_cong_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'emb_cong_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


### 🔑 Key Comparison: `cong_em_id`

In [ ]:
# Discrepancy Check for Key: cong_em_id
if not df_merged_keys.empty:
    col_duck = 'cong_em_id_duck'
    col_ath = 'cong_em_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck != v_ath) & ~(v_duck.isna() & v_ath.isna())
    df_diff_cong_em_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_cong_em_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: cong_em_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_cong_em_id[display_cols].rename(columns={
            col_duck: f'cong_em_id_duckdb',
            col_ath: f'cong_em_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'cong_em_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


### 🔑 Key Comparison: `descong_em_id`

In [ ]:
# Discrepancy Check for Key: descong_em_id
if not df_merged_keys.empty:
    col_duck = 'descong_em_id_duck'
    col_ath = 'descong_em_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck != v_ath) & ~(v_duck.isna() & v_ath.isna())
    df_diff_descong_em_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_descong_em_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: descong_em_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_descong_em_id[display_cols].rename(columns={
            col_duck: f'descong_em_id_duckdb',
            col_ath: f'descong_em_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'descong_em_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


### 🔑 Key Comparison: `trat2_id`

In [ ]:
# Discrepancy Check for Key: trat2_id
if not df_merged_keys.empty:
    col_duck = 'trat2_id_duck'
    col_ath = 'trat2_id_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck != v_ath) & ~(v_duck.isna() & v_ath.isna())
    df_diff_trat2_id = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_trat2_id)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: trat2_id ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_trat2_id[display_cols].rename(columns={
            col_duck: f'trat2_id_duckdb',
            col_ath: f'trat2_id_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'trat2_id' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


### 🔑 Key Comparison: `oocito_embryo_number`

In [ ]:
# Discrepancy Check for Key: oocito_embryo_number
if not df_merged_keys.empty:
    col_duck = 'oocito_embryo_number_duck'
    col_ath = 'oocito_embryo_number_ath'
    
    # Identify mismatch mask (values differ AND not both NaN)
    v_duck = df_merged_keys[col_duck]
    v_ath = df_merged_keys[col_ath]
    
    mismatch_mask = (v_duck != v_ath) & ~(v_duck.isna() & v_ath.isna())
    df_diff_oocito_embryo_number = df_merged_keys[mismatch_mask]
    
    n_diff = len(df_diff_oocito_embryo_number)
    pct_diff = (n_diff * 100.0 / len(df_merged_keys)) if len(df_merged_keys) > 0 else 0.0
    
    print(f"=== KEY ATTRIBUTE: oocito_embryo_number ===")
    print(f"Total oocitos evaluated: {len(df_merged_keys):,}")
    print(f"Total mismatches: {n_diff:,} ({pct_diff:.2f}%)")
    
    if n_diff > 0:
        display_cols = ['oocito_id', 'micro_prontuario_duck', 'micro_data_procedimento_duck', col_duck, col_ath, 'nome_medico_duck']
        df_sample = df_diff_oocito_embryo_number[display_cols].rename(columns={
            col_duck: f'oocito_embryo_number_duckdb',
            col_ath: f'oocito_embryo_number_athena',
            'micro_prontuario_duck': 'prontuario',
            'micro_data_procedimento_duck': 'data_procedimento',
            'nome_medico_duck': 'nome_medico'
        }).head(15)
        display(df_sample)
    else:
        print(f"✅ All matching oocitos have identical values for 'oocito_embryo_number' across DuckDB and Athena.")
else:
    print("Athena dataset unavailable for comparison.")


## 🧬 Part 6: PGT / Genetic Test Results Side-by-Side Comparison

Compares PGT genetic testing outcome distributions (`oocito_resultado_pgd`) side-by-side.
*Note: Empty strings, spaces, and NULL values are normalized together into `'NULL / Sem Teste'`.*


In [ ]:
# 1. Distribution in DuckDB (Normalized NULLs/empty)
query_pgt_duck = '''
SELECT 
    COALESCE(NULLIF(TRIM(oocito_ResultadoPGD), ''), 'NULL / Sem Teste') as resultado_pgt,
    COUNT(*) as count_duck
FROM gold.clinisys_embrioes
GROUP BY 1
'''
df_pgt_duck = duck_con.execute(query_pgt_duck).df()

# 2. Distribution in Athena (Normalized NULLs/empty)
if ath_cur is not None:
    query_pgt_ath = f'''
    SELECT 
        COALESCE(NULLIF(TRIM(oocito_resultado_pgd), ''), 'NULL / Sem Teste') as resultado_pgt,
        COUNT(*) as count_ath
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    GROUP BY 1
    '''
    try:
        ath_cur.execute(query_pgt_ath)
        df_pgt_ath = pd.DataFrame(ath_cur.fetchall(), columns=['resultado_pgt', 'count_ath'])
    except Exception as e:
        print(f"Error querying Athena PGT results: {e}")
        df_pgt_ath = pd.DataFrame()
else:
    df_pgt_ath = pd.DataFrame()

# 3. Side-by-Side Reconciliation Table
if not df_pgt_duck.empty and not df_pgt_ath.empty:
    df_pgt_side = pd.merge(df_pgt_duck, df_pgt_ath, on='resultado_pgt', how='outer').fillna(0)
    df_pgt_side['count_duck'] = df_pgt_side['count_duck'].astype(int)
    df_pgt_side['count_ath'] = df_pgt_side['count_ath'].astype(int)
    df_pgt_side['delta'] = df_pgt_side['count_duck'] - df_pgt_side['count_ath']
    df_pgt_side['match_pct'] = df_pgt_side.apply(
        lambda r: f"{(100.0 * min(r['count_duck'], r['count_ath']) / max(r['count_duck'], r['count_ath'])):.2f}%" if max(r['count_duck'], r['count_ath']) > 0 else '100.00%',
        axis=1
    )
    df_pgt_side = df_pgt_side.sort_values(by='count_duck', ascending=False).reset_index(drop=True)
    df_pgt_side = df_pgt_side.rename(columns={
        'resultado_pgt': 'PGT Result',
        'count_duck': 'DuckDB Count',
        'count_ath': 'Athena Count',
        'delta': 'Delta (Duck - Ath)',
        'match_pct': 'Match %'
    })
    print("=========================================================================")
    print("           PGT RESULT DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE         ")
    print("=========================================================================")
    display(df_pgt_side)


## 🗓️ Part 7: Date Sequence & Timeline Integrity Side-by-Side

Validates chronological integrity and compares date counts side-by-side across procedure dates (`micro_data_procedimento`), freezing dates (`cong_em_data`), and transfer dates (`descong_em_data_transferencia`).


In [ ]:
# Date Sequence Validation
query_date_duck = '''
SELECT 
    COUNT(*) as total_rows,
    COUNT(micro_data_procedimento) as count_data_procedimento,
    COUNT(cong_em_Data) as count_data_congelamento,
    COUNT(descong_em_DataTransferencia) as count_data_transferencia,
    SUM(CASE WHEN CAST(cong_em_Data AS DATE) < CAST(micro_data_procedimento AS DATE) THEN 1 ELSE 0 END) as anomaly_freeze_before_proc,
    SUM(CASE WHEN CAST(descong_em_DataTransferencia AS DATE) < CAST(cong_em_Data AS DATE) THEN 1 ELSE 0 END) as anomaly_transfer_before_freeze
FROM gold.clinisys_embrioes
'''
dict_date_duck = duck_con.execute(query_date_duck).df().iloc[0].to_dict()

if ath_cur is not None:
    query_date_ath = f'''
    SELECT 
        COUNT(*) as total_rows,
        COUNT(micro_data_procedimento) as count_data_procedimento,
        COUNT(cong_em_data) as count_data_congelamento,
        COUNT(descong_em_data_transferencia) as count_data_transferencia,
        SUM(CASE WHEN CAST(cong_em_data AS DATE) < CAST(micro_data_procedimento AS DATE) THEN 1 ELSE 0 END) as anomaly_freeze_before_proc,
        SUM(CASE WHEN CAST(descong_em_data_transferencia AS DATE) < CAST(cong_em_data AS DATE) THEN 1 ELSE 0 END) as anomaly_transfer_before_freeze
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    '''
    try:
        ath_cur.execute(query_date_ath)
        dict_date_ath = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description]).iloc[0].to_dict()
    except Exception as e:
        print(f"Error running Athena date query: {e}")
        dict_date_ath = {}
else:
    dict_date_ath = {}

if dict_date_duck and dict_date_ath:
    date_side_rows = []
    for m in dict_date_duck.keys():
        v_duck = dict_date_duck[m]
        v_ath = dict_date_ath.get(m, 0)
        diff = v_duck - v_ath
        pct = (100.0 * min(v_duck, v_ath) / max(v_duck, v_ath)) if max(v_duck, v_ath) > 0 else 100.0
        date_side_rows.append({
            'Timeline Metric': m,
            'DuckDB (Local)': v_duck,
            'AWS Athena (Prod)': v_ath,
            'Delta (Duck - Ath)': diff,
            'Match %': f"{pct:.2f}%"
        })
    df_date_side = pd.DataFrame(date_side_rows)
    print("=========================================================================")
    print("        DATE SEQUENCE INTEGRITY & COUNTS: DUCKDB VS ATHENA SIDE-BY-SIDE   ")
    print("=========================================================================")
    display(df_date_side)


## 🩺 Part 8: Treatment Outcomes & Cancellation Reasons Side-by-Side

Compares clinical outcome classifications (`trat1_resultado_tratamento`) and non-transfer reasons (`trat1_motivo_nao_transferir`) side-by-side between DuckDB and Athena.
*Note: Empty strings, spaces, and NULL values are normalized together into `'NULL / N/A'`.*


In [ ]:
# 1. Outcomes Side-by-Side (Normalized NULLs/empty)
q_out_duck = '''
SELECT 
    COALESCE(NULLIF(TRIM(trat1_resultado_tratamento), ''), 'NULL / N/A') as resultado_tratamento,
    COUNT(*) as count_duck
FROM gold.clinisys_embrioes
GROUP BY 1
'''
df_out_duck = duck_con.execute(q_out_duck).df()

if ath_cur is not None:
    q_out_ath = f'''
    SELECT 
        COALESCE(NULLIF(TRIM(trat1_resultado_tratamento), ''), 'NULL / N/A') as resultado_tratamento,
        COUNT(*) as count_ath
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    GROUP BY 1
    '''
    try:
        ath_cur.execute(q_out_ath)
        df_out_ath = pd.DataFrame(ath_cur.fetchall(), columns=['resultado_tratamento', 'count_ath'])
    except Exception as e:
        print(f"Error querying Athena outcomes: {e}")
        df_out_ath = pd.DataFrame()
else:
    df_out_ath = pd.DataFrame()

if not df_out_duck.empty and not df_out_ath.empty:
    df_out_side = pd.merge(df_out_duck, df_out_ath, on='resultado_tratamento', how='outer').fillna(0)
    df_out_side['count_duck'] = df_out_side['count_duck'].astype(int)
    df_out_side['count_ath'] = df_out_side['count_ath'].astype(int)
    df_out_side['delta'] = df_out_side['count_duck'] - df_out_side['count_ath']
    df_out_side['match_pct'] = df_out_side.apply(
        lambda r: f"{(100.0 * min(r['count_duck'], r['count_ath']) / max(r['count_duck'], r['count_ath'])):.2f}%" if max(r['count_duck'], r['count_ath']) > 0 else '100.00%',
        axis=1
    )
    df_out_side = df_out_side.sort_values(by='count_duck', ascending=False).reset_index(drop=True)
    df_out_side = df_out_side.rename(columns={
        'resultado_tratamento': 'Treatment Outcome',
        'count_duck': 'DuckDB Count',
        'count_ath': 'Athena Count',
        'delta': 'Delta (Duck - Ath)',
        'match_pct': 'Match %'
    })
    print("=========================================================================")
    print("       TREATMENT OUTCOME DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE     ")
    print("=========================================================================")
    display(df_out_side)

# 2. Non-Transfer Reasons Side-by-Side (Normalized NULLs/empty)
q_cancel_duck = '''
SELECT 
    COALESCE(NULLIF(TRIM(trat1_motivo_nao_transferir), ''), 'NULL / Transferido ou N/A') as motivo_nao_transferir,
    COUNT(*) as count_duck
FROM gold.clinisys_embrioes
WHERE trat1_motivo_nao_transferir IS NOT NULL AND TRIM(trat1_motivo_nao_transferir) != ''
GROUP BY 1
'''
df_cancel_duck = duck_con.execute(q_cancel_duck).df()

if ath_cur is not None:
    q_cancel_ath = f'''
    SELECT 
        COALESCE(NULLIF(TRIM(trat1_motivo_nao_transferir), ''), 'NULL / Transferido ou N/A') as motivo_nao_transferir,
        COUNT(*) as count_ath
    FROM {ATHENA_SCHEMA}.clinisys_embrioes
    WHERE trat1_motivo_nao_transferir IS NOT NULL AND TRIM(trat1_motivo_nao_transferir) != ''
    GROUP BY 1
    '''
    try:
        ath_cur.execute(q_cancel_ath)
        df_cancel_ath = pd.DataFrame(ath_cur.fetchall(), columns=['motivo_nao_transferir', 'count_ath'])
    except Exception as e:
        print(f"Error querying Athena cancel reasons: {e}")
        df_cancel_ath = pd.DataFrame()
else:
    df_cancel_ath = pd.DataFrame()

if not df_cancel_duck.empty and not df_cancel_ath.empty:
    df_cancel_side = pd.merge(df_cancel_duck, df_cancel_ath, on='motivo_nao_transferir', how='outer').fillna(0)
    df_cancel_side['count_duck'] = df_cancel_side['count_duck'].astype(int)
    df_cancel_side['count_ath'] = df_cancel_side['count_ath'].astype(int)
    df_cancel_side['delta'] = df_cancel_side['count_duck'] - df_cancel_side['count_ath']
    df_cancel_side['match_pct'] = df_cancel_side.apply(
        lambda r: f"{(100.0 * min(r['count_duck'], r['count_ath']) / max(r['count_duck'], r['count_ath'])):.2f}%" if max(r['count_duck'], r['count_ath']) > 0 else '100.00%',
        axis=1
    )
    df_cancel_side = df_cancel_side.sort_values(by='count_duck', ascending=False).reset_index(drop=True)
    df_cancel_side = df_cancel_side.rename(columns={
        'motivo_nao_transferir': 'Non-Transfer Reason',
        'count_duck': 'DuckDB Count',
        'count_ath': 'Athena Count',
        'delta': 'Delta (Duck - Ath)',
        'match_pct': 'Match %'
    })
    print("\n=========================================================================")
    print("      NON-TRANSFER REASON DISTRIBUTION: DUCKDB VS ATHENA SIDE-BY-SIDE   ")
    print("=========================================================================")
    display(df_cancel_side)
